## 0. Upload do arquivo
Execute a célula abaixo e selecione o arquivo **BD_veiculos_2.csv** quando solicitado.

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione o arquivo BD_veiculos_2.csv
print("Arquivo carregado:", list(uploaded.keys()))

# Previsão de Preço de Veículos Usados
Regressão Linear Múltipla com dados históricos de transações.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", font_scale=1.05)
fmt = lambda x, _: f"R$ {x:,.0f}"

## 1. Carregamento e Inspeção

In [ ]:
df = pd.read_csv("BD_veiculos_2.csv", sep=";", encoding="utf-8",
                 na_values=["", "NA", "?", "N/A", "-", "null", "None"])

print(f"Shape: {df.shape}")
print("\nNulos por coluna:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.describe().round(2)

**Problemas detectados:**
- `ano`: mínimo 1801, máximo 2073
- `km`: valores negativos e máximo acima de 9 milhões
- `potência do motor`: máximo 45,7 L
- `portas`: valores 0 e 15
- `valor`: mínimo R$ 2,42

## 2. Limpeza e Tratamento de Erros

**2a. Erros de digitação** — padronização dos campos categóricos

In [ ]:
def padronizar(s, mapa):
    s = s.astype(str).str.strip().str.title().replace("Nan", np.nan)
    return s.replace(mapa)

df["combustivel"] = df["combustível"].copy()
df["combustivel"] = padronizar(df["combustivel"],
    {"Flxe": "Flex", "Flexe": "Flex", "Gasoolina": "Gasolina", "Disel": "Diesel"})
df["combustível"] = df["combustivel"]
df.drop(columns=["combustivel"], inplace=True)

df["câmbio"] = padronizar(df["câmbio"],
    {"Manuall": "Manual", "Automatico": "Automático"})

df["direção"] = padronizar(df["direção"],
    {"Hidraulica": "Hidráulica", "Hidráulca": "Hidráulica",
     "Elétrca": "Elétrica", "Eletrica": "Elétrica"})

df["tipo"] = padronizar(df["tipo"],
    {"Seda": "Sedã", "Pick-Up": "Pick-up", "Suv": "SUV"})

df["cor"] = padronizar(df["cor"],
    {"Brnaco": "Branco", "Vermellho": "Vermelho"})

df["marca"] = df["marca"].astype(str).str.strip().str.upper()
df["marca"] = df["marca"].replace(
    {"Fodr": "FORD", "Fiatt": "FIAT", "Hyundaii": "HYUNDAI", "Nan": np.nan})

for col in ["combustível", "câmbio", "direção", "tipo"]:
    print(f"{col}: {sorted(df[col].dropna().unique())}")

**2b. Valores nulos** — mediana para numéricas, moda para categóricas

In [ ]:
for col in ["ano", "km", "potência do motor", "portas", "valor"]:
    df[col].fillna(df[col].median(), inplace=True)

for col in ["tipo", "combustível", "câmbio", "direção", "cor", "marca", "modelo"]:
    df[col].fillna(df[col].mode()[0], inplace=True)

print(f"Nulos restantes: {df.isnull().sum().sum()}")

**2c. Valores fora do domínio**

In [ ]:
n0 = len(df)
df = df[
    df["ano"].between(1950, 2024) &
    df["km"].between(0, 600_000) &
    df["potência do motor"].between(0.8, 7.0) &
    df["portas"].isin([2, 4]) &
    (df["valor"] >= 5_000)
]
print(f"Removidos (domínio): {n0 - len(df)} | Restantes: {len(df)}")

**2d. Inconsistências lógicas**

In [ ]:
n0 = len(df)
mask = (
    ((df["tipo"] == "Hatch") & (df["combustível"] == "Diesel")) |
    ((df["tipo"] == "SUV")   & (df["potência do motor"] < 0.8)) |
    ((df["tipo"] == "Van")   & (df["câmbio"] == "Automático") & (df["potência do motor"] <= 0.9))
)
df = df[~mask]
print(f"Removidos (lógica): {n0 - len(df)} | Shape final: {df.shape}")

## 3. Análise Exploratória (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["valor"], bins=40, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[0].set_title("Distribuição do Preço")
axes[0].set_xlabel("Preço (R$)")
axes[0].set_ylabel("Frequência")
axes[0].tick_params(axis="x", rotation=25)

num_df = df[["ano", "km", "potência do motor", "portas", "valor"]]
sns.heatmap(num_df.corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=axes[1])
axes[1].set_title("Correlação entre Numéricas")

plt.tight_layout()
plt.show()
print(f"Assimetria do preço: {df['valor'].skew():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

order_tipo = df.groupby("tipo")["valor"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="tipo", y="valor", order=order_tipo, palette="muted", ax=axes[0])
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[0].set_title("Preço por Tipo")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.boxplot(data=df, x="câmbio", y="valor", palette="Set2", ax=axes[1])
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[1].set_title("Preço por Câmbio")
axes[1].set_xlabel("")

top = df.groupby("marca")["valor"].median().sort_values(ascending=False).head(8).index
sns.boxplot(data=df[df["marca"].isin(top)], x="marca", y="valor",
            order=top, palette="tab10", ax=axes[2])
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[2].set_title("Preço por Marca (top 8)")
axes[2].set_xlabel("")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df["km"], df["valor"], alpha=0.3, s=15, color="steelblue", edgecolors="none")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[0].set_title("Km x Preço")
axes[0].set_xlabel("Quilometragem")
axes[0].set_ylabel("Preço (R$)")

axes[1].scatter(df["ano"], df["valor"], alpha=0.3, s=15, color="darkorange", edgecolors="none")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[1].set_title("Ano x Preço")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Preço (R$)")

plt.tight_layout()
plt.show()

## 4. Pré-processamento

In [ ]:
df.drop(columns=["placa"], inplace=True)

df_enc = pd.get_dummies(df, drop_first=True)

X = df_enc.drop(columns=["valor"])
y = df_enc["valor"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Features: {X.shape[1]}  |  Treino: {len(X_train)}  |  Teste: {len(X_test)}")

## 5. Treinamento

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred       = lr.predict(X_test_sc)
y_pred_train = lr.predict(X_train_sc)

coefs    = pd.Series(lr.coef_, index=X.columns)
top5_pos = coefs.nlargest(5)
top5_neg = coefs.nsmallest(5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
top5_pos.plot(kind="barh", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[0].set_title("Top 5 — Impacto positivo no preço")

top5_neg.plot(kind="barh", ax=axes[1], color="salmon", edgecolor="white")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[1].set_title("Top 5 — Impacto negativo no preço")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, alpha=0.4, s=18, color="steelblue", edgecolors="none")
lim = [min(y_test.min(), y_pred.min()) * 0.95,
       max(y_test.max(), y_pred.max()) * 1.05]
ax.plot(lim, lim, "r--", lw=1.5, label="y = x (perfeito)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(fmt))
ax.yaxis.set_major_formatter(plt.FuncFormatter(fmt))
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Real (R$)")
ax.set_ylabel("Predito (R$)")
ax.set_title("Real vs. Predito")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Avaliação

In [ ]:
mae   = mean_absolute_error(y_test, y_pred)
rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
r2    = r2_score(y_test, y_pred)
r2_tr = r2_score(y_train, y_pred_train)
n, p  = X_test.shape
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MAE         : R$ {mae:>10,.2f}")
print(f"RMSE        : R$ {rmse:>10,.2f}")
print(f"R2 teste    :    {r2:>10.4f}")
print(f"R2 treino   :    {r2_tr:>10.4f}")
print(f"R2 ajustado :    {r2_adj:>10.4f}")
print(f"Media preco : R$ {y.mean():>10,.2f}")
print(f"Desvio      : R$ {y.std():>10,.2f}")
print(f"RMSE/Desvio :    {rmse/y.std():>10.2%}")

In [ ]:
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(residuos, bins=35, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(0, color="red", linestyle="--", lw=1.5)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[0].set_title("Distribuição dos Resíduos")
axes[0].set_xlabel("Resíduo (R$)")
axes[0].set_ylabel("Frequência")

axes[1].scatter(y_pred, residuos, alpha=0.35, s=15, color="darkorange", edgecolors="none")
axes[1].axhline(0, color="red", linestyle="--", lw=1.5)
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(fmt))
axes[1].set_title("Resíduos vs. Predito")
axes[1].set_xlabel("Predito (R$)")
axes[1].set_ylabel("Resíduo (R$)")

plt.tight_layout()
plt.show()
print(f"Media dos residuos: R$ {residuos.mean():,.2f}")

In [ ]:
X_sc_full = scaler.fit_transform(X)
cv = -cross_val_score(LinearRegression(), X_sc_full, y,
                      cv=10, scoring="neg_root_mean_squared_error")

print(f"CV 10-fold RMSE — Media: R$ {cv.mean():,.2f}  |  Desvio: R$ {cv.std():,.2f}")

plt.figure(figsize=(8, 3))
plt.bar(range(1, 11), cv, color="steelblue", edgecolor="white", alpha=0.85)
plt.axhline(cv.mean(), color="red", linestyle="--", lw=1.5,
            label=f"Media: R$ {cv.mean():,.0f}")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(fmt))
plt.title("RMSE por Fold — CV 10-fold")
plt.xlabel("Fold")
plt.ylabel("RMSE (R$)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Análise e Discussão

**a) Overfitting ou Underfitting?**
Se R² treino ≈ R² teste, o modelo generaliza bem sem overfitting. Valores moderados (0,55–0,75) indicam underfitting leve — a regressão linear não captura toda a não-linearidade do mercado.

**b) Linearidade de km e ano?**
Os scatter plots mostram tendência clara mas com alta dispersão. O padrão de funil nos resíduos (heterocedasticidade) indica que a premissa linear é parcialmente violada — preços altos têm erros maiores.

**c) Impacto dos erros no dataset**

| Tipo | Impacto |
|---|---|
| Nulos | Reduzem o treino efetivo; imputação com mediana introduz viés suave |
| Typos | Criam categorias espúrias no OHE, aumentando dimensionalidade e ruído |
| Fora do domínio | Distorcem a escala e puxam coeficientes para longe do real |
| Inconsistências | Exemplos impossíveis que o modelo tenta erroneamente aprender |

**d) Alta dimensionalidade do OHE**
`marca` e `modelo` geram muitas colunas binárias. Ridge (L2) comprime coeficientes grandes sem zerá-los; Lasso (L1) zera features irrelevantes, funcionando como seletor automático.

**e) Melhorias propostas**
1. Transformação logarítmica do target → normaliza a distribuição assimétrica
2. `idade_veiculo = 2024 - ano` e `km_por_ano = km / idade_veiculo` → features mais intuitivas
3. Ridge/Lasso com GridSearchCV → controle de overfitting
4. Modelos não-lineares (Gradient Boosting) → capturam interações entre features

## 8. Desafio Extra

**8a. Transformação logarítmica do target**

In [ ]:
y_log = np.log(y)
Xtr, Xte, ytr, yte = train_test_split(X, y_log, test_size=0.2, random_state=42)
sc2 = StandardScaler()
m_log = LinearRegression().fit(sc2.fit_transform(Xtr), ytr)
pred_log  = np.exp(m_log.predict(sc2.transform(Xte)))
real_orig  = np.exp(yte)

r2_log   = r2_score(real_orig, pred_log)
rmse_log = np.sqrt(mean_squared_error(real_orig, pred_log))
mae_log  = mean_absolute_error(real_orig, pred_log)

print(f"Original | MAE: R$ {mae:>10,.0f}  RMSE: R$ {rmse:>10,.0f}  R2: {r2:.4f}")
print(f"Log      | MAE: R$ {mae_log:>10,.0f}  RMSE: R$ {rmse_log:>10,.0f}  R2: {r2_log:.4f}")

**8b. Engenharia de features**

In [ ]:
df_e = df.copy()
df_e["idade_veiculo"] = 2024 - df_e["ano"]
df_e["km_por_ano"]    = df_e["km"] / df_e["idade_veiculo"].replace(0, 1)

df_e2 = pd.get_dummies(df_e, drop_first=True)
Xe = df_e2.drop(columns=["valor"])
ye = df_e2["valor"]

Xtr2, Xte2, ytr2, yte2 = train_test_split(Xe, ye, test_size=0.2, random_state=42)
sc3 = StandardScaler()
m_eng = LinearRegression().fit(sc3.fit_transform(Xtr2), ytr2)
pred_eng = m_eng.predict(sc3.transform(Xte2))

r2_eng   = r2_score(yte2, pred_eng)
rmse_eng = np.sqrt(mean_squared_error(yte2, pred_eng))

print(f"Original        | RMSE: R$ {rmse:>10,.0f}  R2: {r2:.4f}")
print(f"+ Eng. features | RMSE: R$ {rmse_eng:>10,.0f}  R2: {r2_eng:.4f}")

**8c. Ridge vs. Lasso com GridSearchCV**

In [ ]:
alphas = {"alpha": [0.01, 0.1, 1, 10, 50, 100, 500, 1000]}

ridge = GridSearchCV(Ridge(), alphas, cv=5, scoring="r2", n_jobs=-1)
ridge.fit(X_train_sc, y_train)

lasso = GridSearchCV(Lasso(max_iter=5000), alphas, cv=5, scoring="r2", n_jobs=-1)
lasso.fit(X_train_sc, y_train)

r2_r  = r2_score(y_test, ridge.best_estimator_.predict(X_test_sc))
r2_l  = r2_score(y_test, lasso.best_estimator_.predict(X_test_sc))
zeros = (lasso.best_estimator_.coef_ == 0).sum()

a_r = ridge.best_params_["alpha"]
a_l = lasso.best_params_["alpha"]

print(f"Linear  | R2: {r2:.4f}")
print(f"Ridge   | R2: {r2_r:.4f}  (alpha={a_r})")
print(f"Lasso   | R2: {r2_l:.4f}  (alpha={a_l}) — {zeros} features zeradas de {X.shape[1]}")